# Stage 3 — DPO Preference Alignment

**Domain:** Finance FAQ Assistant
**Starting point:** the Stage 2 SFT adapter (`outputs/sft_adapter`)
**Data:** `data/preference_dataset.jsonl` (50+ prompt/chosen/rejected triples)
**Goal:** sharpen response *quality* — correctness, helpfulness, tone,
domain-specificity — by teaching the model to prefer the "chosen" answer
over the "rejected" one for the same prompt.

> Run on Colab with a T4 GPU.


In [ ]:
%%capture
!pip install unsloth


In [ ]:
from google.colab import files
import os

if not os.path.exists("data/preference_dataset.jsonl"):
    os.makedirs("data", exist_ok=True)
    print("Upload preference_dataset.jsonl:")
    uploaded = files.upload()
    for f in uploaded:
        os.rename(f, f"data/{f}")

# Also make sure outputs/sft_adapter/ from Stage 2 is present in this runtime
# (upload the folder as a zip and unzip it, or re-clone your repo after
# pushing the Stage 2 adapter, or re-run Stage 2 in this same session).


In [ ]:
from unsloth import FastLanguageModel
import torch, os

max_seq_length = 2048
SFT_ADAPTER = "outputs/sft_adapter"
assert os.path.isdir(SFT_ADAPTER), "Stage 2 SFT adapter not found — load it before running DPO."

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_ADAPTER,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)


In [ ]:
from unsloth import PatchDPOTrainer
PatchDPOTrainer()

from trl import DPOTrainer, DPOConfig


## Format the preference dataset

`DPOTrainer` expects a dataset with `prompt`, `chosen`, and `rejected` text
columns, using the same prompt template as Stage 2 so the model sees a
consistent format across SFT and DPO.


In [ ]:
import json
from datasets import Dataset

PROMPT_TEMPLATE = """Below is a question about personal finance. Write a clear, accurate, and helpful response.

### Question:
{instruction}

### Response:
"""

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

raw_rows = load_jsonl("data/preference_dataset.jsonl")
print(f"Loaded {len(raw_rows)} preference triples")

def format_example(example):
    return {
        "prompt": PROMPT_TEMPLATE.format(instruction=example["prompt"]),
        "chosen": example["chosen"] + tokenizer.eos_token,
        "rejected": example["rejected"] + tokenizer.eos_token,
    }

dpo_dataset = Dataset.from_list(raw_rows).map(format_example)


In [ ]:
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,          # Unsloth shares the frozen base weights as the reference
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 5e-6,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs/dpo",
        report_to = "none",
        beta = 0.1,
        max_length = max_seq_length,
        max_prompt_length = 512,
    ),
    train_dataset = dpo_dataset,
    processing_class = tokenizer,
)


In [ ]:
dpo_trainer.train()


In [ ]:
model.save_pretrained("outputs/dpo_adapter")
tokenizer.save_pretrained("outputs/dpo_adapter")
print("Saved Stage 3 (DPO) adapter to outputs/dpo_adapter")

# Optional: also save a merged, full-precision model for easy distribution /
# use with src/inference.py without needing PEFT at load time.
model.save_pretrained_merged(
    "outputs/dpo_merged", tokenizer, save_method = "merged_16bit",
)


## Final sanity check

Compare this answer against the Stage 2 (SFT-only) answer for the same
question — this is the pair you'll record in `reports/final_evaluation.md`.


In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=200):
    prompt = PROMPT_TEMPLATE.format(instruction=question)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Response:")[-1].strip()

print(ask("How can I start building an emergency fund?"))
